# പാഠം 13 - ഏജന്റ് മെമ്മറി


## സജ്ജീകരണം

ഈ നോട്ട്‌ബുക്ക് **Microsoft Agent Framework** (MAF) ഉപയോഗിച്ച് **സ്ഥിരമായ സ്മരണ** ഉള്ള ഒരു യാത്ര ബുക്കിംഗ് ഏജന്റ് എങ്ങനെ നിർമ്മിക്കാമെന്ന് പ്രത്യക്ഷപ്പെടുത്തുന്നു.

ഏജന്റിന്റെ ചിന്തന, ചെറുകാല, ദീർഘകാല എന്നിവയടക്കം വ്യത്യസ്ത തരത്തിലുള്ള സ്മരണകൾ എങ്ങനെ ഏജന്റ് സംഭാഷണങ്ങളിൽ വിവരങ്ങൾ നിലനിർത്താനും ഉപയോഗിക്കാനും ബാധിക്കുന്നു എന്ന് നിങ്ങൾ പഠിക്കും.

**ആവശ്യമായ മുൻകൂർ വിദ്യകൾ:**
- ഡിസ്പ്ലേ ചെയ്ത ഒരു ചാറ്റ് മോഡൽ (ഉദാഹരണത്തിന് `gpt-4.1-mini`) ഉളള Microsoft Foundry പ്രോജക്റ്റ്.
- Azure CLIൽ ലോഗിൻ ചെയ്തിരിക്കണം — നിങ്ങളുടെ ടെർമിനലിൽ `az login` പ്രവർത്തിപ്പിക്കുക.
- `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Microsoft Foundry പ്രോജക്റ്റ് എൻഡ്‌പോയിന്റ്.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങളുടെ ഡിപ്പ്ലോയ്മെന്റ് ചെയ്ത മോഡലിന്റെ പേര്.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ഏജന്റ് മെമ്മറിയുടെ തരംകൾ

എ.ഐ. ഏജന്റുകൾ വെറൈറ്റി മെമ്മറി തരംകൾ ഉപയോഗിച്ചുകൊണ്ട് വ്യത്യസ്ത ആവശ്യങ്ങൾ നിറവേറ്റുന്നു:

### വാർക്കിംഗ് മെമ്മറി
സംവാദ ത്രെഡ് തന്നെ — ഒരേ സെഷനിൽ കൈമാറുന്ന സന്ദേശങ്ങൾ. ഏജന്റ് സംവാദത്തിന്റെ മുൻസന്ദേശങ്ങൾക്ക് തിരിച്ചുപോകാൻ കഴിയും തുടർച്ച നിലനിർത്തുവാൻ. MAF-ൽ ഇത് **`agent.create_session()`** വഴി സൃഷ്ടിക്കപ്പെടുന്നു, ഇത് `AgentSession` ആണ് തിരികെ നൽകുന്നത്.

### ചുരുങ്ങിയകാല മെമ്മറി
ഒരു ടാസ്‌ക് അല്ലെങ്കിൽ സെഷൻ സമയയളവിനുള്ളിൽ നിലനിർത്തുന്ന വിവരങ്ങൾ, സ്ഥിരമായി സൂക്ഷിച്ചിരിക്കുന്നതല്ല. ഉദാഹരണത്തിന്, ഏജന്റ് മൾട്ടി-ടേൺ പദ്ധതിയിടൽ സംവാദത്തിൽ സംഘടിപ്പിച്ച അവസ്ഥകൾ ശേഖരിച്ച് അന്തിമ യാത്രാപദ്ധതി സൃഷ്ടിക്കാൻ ഉപയോഗിക്കും.

### ദീർഘകാല മെമ്മറി
**സെഷനുകൾക്ക് മുകളിലാണ്** നിലനിർത്തുന്നത് ഇഷ്ടങ്ങളും വസ്തുതകളും. തിരിച്ചുവരുന്ന ഉപയോക്താവ് ഭക്ഷണപരമായ നിയന്ത്രണങ്ങൾ അല്ലെങ്കിൽ യാത്ര ശൈലി ആവർത്തിക്കേണ്ടതില്ല. ദീർഘകാല മെമ്മറി സാധാരണയായി ഒരു ബാഹ്യ സ്റ്റോർ — ഡാറ്റാബേസ്, ഫയൽ, അല്ലെങ്കിൽ വെക്ടർ സൂചിക — സഹായത്തോടെ ഏജന്റിന് ടൂൾസ് വഴി ലഭ്യമാക്കുന്നു.


## സെഷനുകളോടെയുള്ള പ്രവർത്തന ഓര്‍മ്മ

ഓർമ്മയുടെ ഏറ്റവും ലളിതമായ രൂപം സംവാദ സെഷനാണ്. നിങ്ങൾ ഒരേ സെഷൻ (`agent.create_session()` ഉപയോഗിച്ച് സൃഷ്ടിച്ച) തുടര്‍ച്ചയായ `agent.run()` കാൾസ് ਨੂੰ പാസ്സ് ചെയ്താൽ, ഏജന്റ് ആ സംവാദത്തിന്റെ പൂർണ്ണ ചരിത്രവും കാണുകയും മുമ്പത്തെ വിവരങ്ങൾ ഓർമ്മിക്കാനും കഴിയും.

ഒരു യാത്ര ഏജന്റ് സൃഷ്ടിച്ച് പ്രവർത്തന ഓർമ്മ തെളിയിക്കാമെന്ന് നോക്കാം.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ഏജന്റ് ബജറ്റ് ശരിയായി ഓർമ്മിച്ച് വെച്ചത് എങ്കിൽ രണ്ട് സന്ദേശങ്ങളും ഒരേ സെഷനിലേക്ക് പെട്ടതിനാൽ ആണ്. ഇതാണ് **കാർമ്മിക സ്മരണം** — ഇത് സെഷന്റെ ഓർമ്മ കാലയളവിൽ മാത്രം നിലവിലുണ്ട്.

### പുതിയ ത്രെഡിൽ എന്താകും സംഭവിക്കുന്നത്?

നാം **പുതിയ** സെഷൻ സൃഷ്ടിച്ചാൽ, ഏജന്റിന് മുൻ സംഭാഷണത്തിന്റെ ഓർമ്മയില്ല:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ദീർഘകാല ഓർമ്മ മാതൃക

ഉപയോക്താവിന്റെ മുൻഗണനകൾ **സെഷനുകളിലുടനീളം** ഓർമ്മിക്കാൻ, സംഭാഷണ ത്രെഡിന് പുറത്തായി നിലനിൽക്കുന്ന സ്ഥിരമായ ഒരു സ്റ്റോർ ആവശ്യമാണ്. ഒരു ഏജന്റ് ഈ സ്റ്റോർ ആക്‌സസ് ചെയ്യുന്നത് **ടൂളുകൾ** വഴിയാണ് — ഇതിൽ നിന്ന് വിവരങ്ങൾ സേവ് ചെയ്യാനും വീണ്ടെടുക്കാനും അത് ഉപയോഗിക്കാൻ കഴിയുന്ന പ്രവർത്തനങ്ങൾ.

താഴെ നാം ഒരു ലളിതമായ ഇൻ-മെമ്മറി മുൻഗണന സ്റ്റോർ ഇൻപ്ലിമെന്റ് ചെയ്യുന്നു (തയാറാവുമ്പോൾ നിങ്ങൾ അതിനെ ഒരു ഡാറ്റാബേസ് അല്ലെങ്കിൽ വെക്ടർ ഇൻഡക്സ് ഉപയോഗിച്ച് പിന്തുണയ്ക്കും) കൂടാതെ ഇത് ഏജന്റ് ഉപയോഗിക്കാവുന്ന ടൂളുകളായി വെക്കുന്നു.

### ആർക്കിടെക്ചർ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### രംഗം 1 — ആദ്യമായുള്ള ഉപയോക്താവ് വേളാച്ചെ കുറിച്ചുള്ള യാത്ര بک്‌ചെയ്യുന്നു

സാറ პირველമായി സന്ദര്‍ശിക്കുന്നു. ഏജന്റ് ഉപകരണങ്ങള്‍ വഴി അവളുടെ ഇഷ്ടങ്ങള്‍ സൂക്ഷിച്ച് അവ ഹോട്ടലുകള്‍ ശുപാര്‍ശ ചെയ്യാന്‍ ഉപയോഗിക്കണം.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ദൃശ്യവൽക്കരണം 2 — സാറാ ആഴ്ചകൾക്ക് ശേഷം മടങ്ങി വരുന്നു

സാറാ ഒരു **പുതിയ ത്രെഡ്** തുടങ്ങുന്നു (പുതിയ സെഷൻ അനുകരിക്കുന്നു). പ്രവർത്തനസ്മൃതി ശൂന്യമാണ്, എന്നാൽ ദീർഘകാല ഇഷ്ടങ്ങൾ സൂക്ഷിക്കുന്ന വേദി ഇപ്പോഴും അവളുടെ വിവരങ്ങൾ ഉണ്ട്. ഏജന്‍റ് അത് തിരിച്ച് കൊണ്ടുവന്നു ശിപാരസുകൾ വ്യക്തിഗതമാക്കാൻ ഉപയോഗിക്കണം.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ മൂന്ന്നോ ചെയ്യേണ്ട ഏജന്റ് സ്മരണയുടെ തരങ്ങളും അതികോഡ് എങ്ങനെ Microsoft Agent Framework ഉപയോഗിച്ച് നടപ്പിലാക്കാമെന്നതും പഠിച്ചു:

| സ്മരണ തരം | MAF യന്ത്രം | ആയുസ്സ് |
|---|---|---|
| **പ്രവർത്തനമായ** | `agent.create_session()` | ഒറ്റ സംവാദം |
| **ഷോർട്ട്-ടേം** | ദ.Thread-ൽ സങ്കലിതമായ സന്ദർഭം | ഒറ്റ ടാസ്ക് / സെഷൻ |
| **ദീർഘകാലം** | `@tool` ഫങ്ഷനുകൾ വഴി പുറം സംഭരണിയിൽ அணുക്കുന്നതു | സെഷനുകൾക്ക് മുകളിൽ |

### മുഖ്യ വർഗ്ഗീകരണങ്ങൾ
1. **`agent.create_session()`** പ്രവർത്തനമായ സ്മരണ നൽകുന്നു — ഏജന്റ് ഒരു സെഷനിലെ മുഴുവൻ സംവാദ ചരിത്രവും കാണുന്നു.
2. **പുതിയ സെഷനുകൾ സന്ദർഭം നഷ്ടപ്പെടുത്തുന്നു** — ദീർഘകാല സ്മരണ ഇല്ലെങ്കിൽ ഏജന്റ് മുൻസഞ്ചാരങ്ങളെ ഓർക്കാനാവില്ല.
3. **`@tool` ഫങ്ഷനുകൾ** ഇടവേള പുനരുപയോഗം ചെയ്യുന്നു — ഈ ഫങ്ഷനുകൾ ഏജന്റിന് സ്ഥിരമായ സ്റ്റോറിൽ വിവരങ്ങൾ സേവ് ചെയ്യാനും തിരിച്ചറിയാനുമാകും.
4. ** വ്യക്തിഗതവൽക്കരണം കാലക്രമേണ മെച്ചപ്പെടുന്നു** — കൂടുതൽ മുൻഗണനകൾ സൂക്ഷിച്ചാൽ ഏജന്റിന്റെ ശുപാർശകൾ ഉന്നത നിലവാരത്തിൽ olur.

### യാഥാർത്ഥ്യ പ്ലവനങ്ങൾ
- **ഉപഭോക്തൃ സേവനം**: ഉപഭോക്താവിന്റെ ചരിത്രവും മുൻഗണനകളും ഓർക്കുക
- **വ്യക്തിഗത സഹായി**: ദിനങ്ങൾ կամ ആഴ്ചകളിലെ സന്ദർഭം നിലനിർത്തുക
- **ആരോഗ്യം**: രോഗിയുടെ വിവരങ്ങളും മുൻഗണനകളും പിന്തുടരുക
- **ഇ-കൊമേഴ്സ്**: ചരിത്രത്തെ അടിസ്ഥാനമാക്കി വ്യക്തിഗത ഷോപ്പിങ്

### പിന്‍വര്‍മകള്‍
- ഇനിയുള്ള dict-നെ ഡാറ്റാബേസ് അല്ലെങ്കിൽ വെക്ടർ സ്റ്റോർ (ഉദാ: Azure AI Search) കൊണ്ടു മാറ്റുക
- സമയസംവേദനാപരമായ വിവരങ്ങൾക്ക് സ്മരണ കാലഹരണപ്പെടുക ചേർക്കുക
- പങ്കുവെച്ച സ്മരണയുള്ള മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ നിർമ്മിക്കുക
- ജ്ഞാന-ഗ്രാഫ് പിന്തുണയുള്ള സ്മരണയ്ക്ക് Cognee നോട്ട്ബുക്ക് പരിശോധിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
